In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import tensor

import numpy as np
import pandas as pd
import networkx as nx

In [2]:
DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'

# Aims outline

Use same as first attempt but replace linear function with MML functon from LEMBAS

# Using network to filter input TFs to those relevant to the target node of a given model

In [ ]:
#creating network for network TSV, obtain all nodes in all simple paths to the target not specified
def genes_in_path(graph, source_nodes, target_node):
    '''
    Paramaters
    -------------
    parameter : graph
        networkx graph
    parameter : soruce_nodes
        list of node labels you wish to use as source nodes
    target_node : string
        label of desired target node
    
    Returns
    -------------
    nodes : List
        List of node labels in the network that are present in at least on shortest path to the specified target node from any of the source nodes.
    '''
    #keep track of all unique nodes found in a shortest path
    node_set = set()
    for source_node in source_nodes:
        #catch errors where there a no path from source to target
        try:
            sps = list(nx.all_shortest_paths(graph, 
                            source=source_node, 
                            target=target_node,
                            weight = None))
        except nx.NetworkXNoPath:
            print(f'No path from {source_node} to {target_node}')

        for path_nodes in sps:
            node_set.update(set(path_nodes))

    return(list(node_set))


In [4]:
net = pd.read_csv(f"{DATA_ROOT}/Full data files/network(full).tsv", sep='\t')

In [5]:
network = nx.from_pandas_edgelist(net, 
                             source='TF', 
                             target='Gene', 
                             edge_attr='Interaction')

In [6]:
gene_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

In [7]:
TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)

In [12]:
print(TF_expressions.shape)
print(gene_expressions.shape)

(15935, 1198)
(15935, 16101)


In [13]:
#subset TFs and genes to only those that are present in the network
TF_expressions = TF_expressions[[gene for gene in TF_expressions.columns if gene in list(network.nodes)]]
gene_expressions = gene_expressions[[gene for gene in gene_expressions.columns if gene in list(network.nodes)]] 

In [14]:
print(TF_expressions.shape)
print(gene_expressions.shape)

(15935, 1197)
(15935, 16100)


In [16]:
genes_to_keep = genes_in_path(network, list(TF_expressions.columns), 'AAGAB')

No path from DBX1 to AAGAB
No path from ZNF512B to AAGAB


In [17]:
TF_expressions = TF_expressions[[gene for gene in TF_expressions.columns if gene in list(genes_to_keep)]]

### Creating dataset object

In [11]:
#torch tutorial code to use accelerator when available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
#DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
#TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)
#TF_expressions.shape

In [ ]:
from torch.utils.data import Dataset


class CustomTFGE(Dataset):
    def __init__(self, device, TF_expressions, gene_expressions, target_gene, network, transform=None, target_transform=None):
        '''
        Custom dataset for loading expression data across all TFs and samples and one target gene expression across all samples
        For a given index will return all TF expressions in one sample and target gene expression across all samples

        Parameters
        --------------
        device : torch device
            torch device to put dataset on, must be the same device as the model
        TF_expressions : Pandas dataframe
            pandas dataframe of TF expressions where columns are gene names and rows are samples. Expressions should be TPM normalised and log10 transformed.
        gene_expressions : Pandas dataframe
            pandas dataframe of target gene expressions, columns are gene names and rows are samples. Expressions should be TPM normalised and log10 transformed.
        target_gene : String
            String containing name of target gene for given model.
        network : networkx network
            Undirected network of the singalling pathway of interest. Used to filter TFs used when predicting target gene expression.
        '''

        #load the two tsv files
        self.target_gene = target_gene
        self.TF_expressions = TF_expressions
        self.gene_expressions = gene_expressions[self.target_gene]

        #no transforms needed so set to none
        self.transform = transform
        self.target_transform = target_transform

        #convert to torch tensors
        self.TF_expressions = torch.tensor(np.asarray(self.TF_expressions).T, dtype = torch.float32, device = device)
        self.gene_expressions = torch.tensor(np.asarray(self.gene_expressions), dtype = torch.float32, device = device)

    def __len__(self):
        #length of the dataset is the number of samples (not TFs in the dataset) - 15935
        return self.TF_expressions.shape[0]

    def __getitem__(self, idx):
        #in this case want to always retrieve the same gene (target) but a different sample containg all the TF values
        TFs_exp = self.TF_expressions[:, idx]
        Gene_exp = self.gene_expressions
        #returns TFs exp and Gene_exp 
        return TFs_exp, Gene_exp

In [14]:
#initialise an instance of the dataset object - pass device so tensors and model on same device
dataset = CustomTFGE(device, TF_expressions=TF_expressions, gene_expressions=gene_expressions, target_gene = 'AADAT')
dataset

In [15]:
#new way to create train and test dataset with pytorches dataset objects
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [ ]:
#defining the neural network
class BasicNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #self.flatten = nn.Flatten()
        self.linear_layer = nn.Sequential(
            #linear activation layer takes 1198 TFs per sample -> outputs a value for the TF for all 15935 samples
            nn.Linear(1198, 15935)
        )
    
    def forward(self, x):
        #forward pass is simply the linear layer
        expressions = self.linear_layer(x)
        return(expressions)

    

In [17]:
#put model on same device as the tensors
model = BasicNeuralNetwork().to(device)
print(model)

BasicNeuralNetwork(
  (linear_layer): Sequential(
    (0): Linear(in_features=1198, out_features=15935, bias=True)
  )
)


## Train test loop

In [18]:
def train_loop(dataloader, model, loss_fn, optimizer):
    losses = []
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        loss = loss.item()
        print(f"loss: {loss:>7f}")
        losses.append(loss)
    return(losses)


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            print(f"MSE on test dataset is {loss_fn(pred, y).item()}")
            


In [19]:
#intialise hyperparameters - batch size is number of samples so that each backprop is done with the entire dataset (gradient descent not stochastic gradient descent)
#dataset is small enough for this to be fine
#investigate hyperparam tuning later
learning_rate = 1e-3
batch_size = 15935
epochs = 500

#initialize MSE loss function - same as LEMBAS
loss_fn = nn.MSELoss()

#initialise same optimiser as LEMBAS
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [20]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [21]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    losses = train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.272735
MSE on test dataset is 0.469354510307312
Epoch 2
-------------------------------
loss: 0.484184
MSE on test dataset is 0.2672496736049652
Epoch 3
-------------------------------
loss: 0.266905
MSE on test dataset is 0.36422058939933777
Epoch 4
-------------------------------
loss: 0.335391
MSE on test dataset is 0.46842652559280396
Epoch 5
-------------------------------
loss: 0.422446
MSE on test dataset is 0.46693646907806396
Epoch 6
-------------------------------
loss: 0.421651
MSE on test dataset is 0.3963909149169922
Epoch 7
-------------------------------
loss: 0.362846
MSE on test dataset is 0.3183616101741791
Epoch 8
-------------------------------
loss: 0.299322
MSE on test dataset is 0.2604418992996216
Epoch 9
-------------------------------
loss: 0.254290
MSE on test dataset is 0.2191619575023651
Epoch 10
-------------------------------
loss: 0.223371
MSE on test dataset is 0.19025219976902008
Epoch 11
-----------------

# Saving model

In [ ]:
torch.save(model, 'models/first_attempt_single_target_model.pth')

In [ ]:
#How to load for reference
model = torch.load('models/first_attempt_single_target_model.pth', weights_only=False)